# ReLU variants: Leaky ReLU, PReLU, ELU and SELU
Dead ReLU nodes made on purpose (large learning rate, negative bias), the variants that avoid them,
PReLU's learned slopes and SELU's self-normalisation.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_moons

tf.config.experimental.enable_op_determinism()   # same numbers on every run

## The functions and their derivatives (saved for Figure 1)
Leaky ReLU slope: 0.01 in the original definition, 0.2 for Keras' "leaky_relu" string, 0.3 for the
LeakyReLU layer; here 0.1 so the slope is visible in the figure. Keras' constants:
ELU alpha 1, SELU alpha 1.6733 and lambda 1.0507.

In [2]:
ALPHA, LAMBDA = 1.6732632423543772, 1.0507009873554805
z = np.linspace(-4, 3, 701)
elu = np.where(z > 0, z, np.exp(z) - 1)
f = pd.DataFrame({
    "z": z,
    "relu": np.maximum(0, z), "d_relu": (z > 0) * 1.0,
    "leaky": np.where(z > 0, z, 0.1 * z), "d_leaky": np.where(z > 0, 1, 0.1),
    "elu": elu, "d_elu": np.where(z > 0, 1, elu + 1),                  # ELU' = ELU(z) + alpha for z < 0
    "selu": LAMBDA * np.where(z > 0, z, ALPHA * (np.exp(z) - 1)),
    "d_selu": LAMBDA * np.where(z > 0, 1, ALPHA * np.exp(z))})
f.to_csv("data/variant_curves.csv", index=False)
print("Keras SELU at z = -1 and 1:", keras.activations.selu(np.array([-1.0, 1.0])).numpy().round(4))
print("ours:                      ", f.loc[f.z.round(2).isin([-1.0, 1.0]), "selu"].round(4).tolist())
print("lowest values: ELU %.3f, SELU %.3f" % (f.elu.min(), f.selu.min()))

Keras SELU at z = -1 and 1: [-1.1113  1.0507]
ours:                       [-1.1113, 1.0507]
lowest values: ELU -0.982, SELU -1.726


## Making ReLU nodes die
500 moons points, standardised. Two hidden layers of 32 nodes, plain SGD, 200 epochs.
A node is dead when its weighted sum z is negative for every training row: it outputs 0 for all of them.

In [3]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=0)
X = ((X - X.mean(0)) / X.std(0)).astype("float32")

def build(act, lr, bias, seed=0):
    keras.utils.set_random_seed(seed)
    hidden = lambda: keras.layers.Dense(32, activation=act,
                                        bias_initializer=keras.initializers.Constant(bias))
    m = keras.Sequential([keras.Input(shape=(2,)), hidden(), hidden(),
                          keras.layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=lr), loss="binary_crossentropy",
              metrics=["accuracy"])
    return m

def negative_everywhere(m):
    # share of nodes in each hidden layer whose z < 0 on every row
    out, h = [], X
    for layer in m.layers[:2]:
        z = h @ layer.kernel.numpy() + layer.bias.numpy()
        out.append(float(np.mean((z <= 0).all(axis=0))))
        h = layer(h).numpy()
    return out

runs = [("ReLU, learning rate 0.1", "relu", 0.1, 0.0),
        ("ReLU, learning rate 10", "relu", 10.0, 0.0),
        ("ReLU, bias -1", "relu", 0.1, -1.0),
        ("ReLU, bias +0.01, learning rate 10", "relu", 10.0, 0.01),
        ("Leaky ReLU, bias -1", "leaky_relu", 0.1, -1.0),
        ("ELU, bias -1", "elu", 0.1, -1.0)]
rows, models = [], {}
for name, act, lr, bias in runs:
    m = build(act, lr, bias)
    before = negative_everywhere(m)
    w0 = [l.kernel.numpy().copy() for l in m.layers[:2]]
    h = m.fit(X, y, epochs=200, batch_size=32, verbose=0)
    after = negative_everywhere(m)
    models[name] = (m, w0)
    rows.append(dict(run=name, layer1_before=before[0], layer2_before=before[1],
                     layer1_after=after[0], layer2_after=after[1], accuracy=h.history["accuracy"][-1]))
res = pd.DataFrame(rows)
res.to_csv("data/dead_nodes.csv", index=False)
print(res.round(2).to_string(index=False))

                               run  layer1_before  layer2_before  layer1_after  layer2_after  accuracy
           ReLU, learning rate 0.1           0.00           0.06          0.00          0.09      0.97
            ReLU, learning rate 10           0.00           0.06          0.41          0.69      0.51
                     ReLU, bias -1           0.72           1.00          0.72          1.00      0.50
ReLU, bias +0.01, learning rate 10           0.00           0.06          0.75          1.00      0.45
               Leaky ReLU, bias -1           0.72           1.00          0.44          0.47      0.89
                      ELU, bias -1           0.72           0.97          0.38          0.34      0.96


## When do the nodes die, and do they come back?
The learning-rate-10 network again, one epoch at a time: the share of dead nodes after epochs 1, 2, 3
and 200. Then 20 more epochs, checking the weights into the dead first-layer nodes.

In [4]:
m = build("relu", 10.0, 0.0)
for epoch in range(1, 201):
    m.fit(X, y, epochs=1, batch_size=32, verbose=0)
    if epoch in (1, 2, 3, 200):
        print(f"after epoch {epoch:3d}: dead share per layer {np.round(negative_everywhere(m), 2)}")
z = X @ m.layers[0].kernel.numpy() + m.layers[0].bias.numpy()
dead = (z <= 0).all(axis=0)
w_before = m.layers[0].kernel.numpy().copy()
m.fit(X, y, epochs=20, batch_size=32, verbose=0)
change = np.abs(m.layers[0].kernel.numpy() - w_before)
print("dead nodes in layer 1:", dead.sum(), "of 32")
print("largest change of a weight into a dead node over 20 more epochs:", change[:, dead].max())

after epoch   1: dead share per layer [0.41 0.69]


after epoch   2: dead share per layer [0.41 0.69]


after epoch   3: dead share per layer [0.41 0.69]


after epoch 200: dead share per layer [0.41 0.69]


dead nodes in layer 1: 13 of 32
largest change of a weight into a dead node over 20 more epochs: 0.0


## PReLU learns its own slope
`keras.layers.PReLU` learns one slope a per node (it starts at 0).

In [5]:
keras.utils.set_random_seed(0)
prelu = keras.Sequential([keras.Input(shape=(2,)),
                          keras.layers.Dense(8), keras.layers.PReLU(),
                          keras.layers.Dense(1, activation="sigmoid")])
prelu.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
h = prelu.fit(X, y, epochs=100, batch_size=32, verbose=0)
prelu.summary()
print("learned slopes a:", prelu.layers[1].get_weights()[0].round(3))
print("training accuracy:", round(h.history["accuracy"][-1], 3))

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_21 (Dense)                │ (None, 8)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ p_re_lu (PReLU)                 │ (None, 8)              │             8 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125 (504.00 B)

 Trainable params: 41 (164.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 84 (340.00 B)

learned slopes a: [ 3.234  0.005 -1.458 -0.846  3.443  0.248 -1.12  -1.221]
training accuracy: 0.972


## SELU keeps mean 0 and standard deviation 1 through many layers
30 layers of 256 nodes, no training. Weights drawn with standard deviation sqrt(1 / fan_in)
(LeCun normal, the initialisation SELU is designed for). Inputs: 1000 rows of standard normal numbers.

In [6]:
rng = np.random.default_rng(0)
acts = {"SELU": lambda v: LAMBDA * np.where(v > 0, v, ALPHA * (np.exp(v) - 1)),
        "ELU": lambda v: np.where(v > 0, v, np.exp(v) - 1),
        "ReLU": lambda v: np.maximum(0, v)}
stats = []
for name, g in acts.items():
    rng = np.random.default_rng(0)
    a = rng.standard_normal((1000, 256))
    for layer in range(1, 31):
        W = rng.standard_normal((256, 256)) * np.sqrt(1 / 256)
        a = g(a @ W)
        stats.append(dict(activation=name, layer=layer, mean=a.mean(), std=a.std()))
stats = pd.DataFrame(stats)
stats.to_csv("data/selu_layers.csv", index=False)
print(stats[stats.layer.isin([1, 10, 30])].round(3).to_string(index=False))

activation  layer  mean   std
      SELU      1 0.001 1.004
      SELU     10 0.006 1.022
      SELU     30 0.002 1.004
       ELU      1 0.162 0.791
       ELU     10 0.016 0.249
       ELU     30 0.002 0.085
      ReLU      1 0.401 0.588
      ReLU     10 0.019 0.028
      ReLU     30 0.000 0.000
